# CS3081 Artificial Intelligence — Lab 4
## Bayesian Network Lab: Likelihood and Inference
- **Group Members:**
  - Ali Ahmed (S24109004)
  - Abubaker Baobaid (S23108655)
  - Abdulrahman Al Shalhoub (S24109270)
- **Instructor:** Dr. Naila Marir
- **Semester:** Fall 2026
- **Department:** Computer Science, Effat College of Engineering

### Step 0: Set up VS Code
**Check Question:** How do you run a cell and move to the next one?  
**Answer:** Press Shift + Enter

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pgmpy
print("pgmpy version:", pgmpy.__version__)

pgmpy version: 1.1.2


---
### Step 1: Meet the network
We model a small world with 4 variables:
- **Rain:** `none`, `light`, `heavy`
- **Maintenance:** `yes`, `no` (depends on Rain)
- **Train:** `on time`, `delayed` (depends on Rain and Maintenance)
- **Appointment:** `attend`, `miss` (depends on Train)

**Question 1:** Which nodes are the parents of Train?  
**Answer:** Rain and Maintenance

**Question 2:** Does Appointment depend directly on Rain?  
**Answer:** No, only through Train

---
### Step 2: Build the network in code
Each node has a Conditional Probability Table (CPT). Every column adds up to 1.

In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

# 2) P(Rain): no parents, so one column
rain = TabularCPD(
    variable="rain",
    variable_card=3,
    values=[[0.7],  # none
            [0.2],  # light
            [0.1]], # heavy
    state_names={"rain": ["none", "light", "heavy"]}
)

# 3) P(Maintenance | Rain): one column for each value of rain
#                      rain = none  light  heavy
maintenance = TabularCPD(
    variable="maintenance",
    variable_card=2,
    values=[[0.4, 0.2, 0.1],  # yes
            [0.6, 0.8, 0.9]], # no
    evidence=["rain"],
    evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]}
)

# 4) P(Train | Rain, Maintenance): one column for each (rain, maintenance) pair
#                           (none,yes) (none,no) (light,yes) (light,no) (heavy,yes) (heavy,no)
train = TabularCPD(
    variable="train",
    variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],  # on time
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]], # delayed
    evidence=["rain", "maintenance"],
    evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]}
)

# 5) P(Appointment | Train): one column for each value of train
#                                train = on time  delayed
appointment = TabularCPD(
    variable="appointment",
    variable_card=2,
    values=[[0.9, 0.6],  # attend
            [0.1, 0.4]], # miss
    evidence=["train"],
    evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]}
)

# 6) Put the tables into the graph and check them
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


**Question 3:** From the code, what is P(maintenance = yes | rain = light)?  
**Answer:** **0.2**

**Question 4:** From the code, what is P(train = delayed | rain = heavy, maintenance = no)?  
**Answer:** **0.5**

**Question 5:** How many numbers are in the Train table in total?  
**Answer:** **12** (2 rows x 6 columns)

---
### Step 3: Compute a joint probability (likelihood)
Situation: no rain, no maintenance, train on time, you attend.
$$P(r, m, t, a) = P(r) \times P(m \mid r) \times P(t \mid r, m) \times P(a \mid t)$$

**Question 6:** Four numbers from the tables:
- $P(\text{none}) = 0.7$
- $P(\text{no} \mid \text{none}) = 0.6$
- $P(\text{on time} \mid \text{none}, \text{no}) = 0.9$
- $P(\text{attend} \mid \text{on time}) = 0.9$

**Question 7:** Multiply them:
$$0.7 \times 0.6 \times 0.9 \times 0.9 = \mathbf{0.3402}$$

In [3]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="none") *
     maintenance.get_value(maintenance="no", rain="none") *
     train.get_value(train="on time", rain="none", maintenance="no") *
     appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


---
### Step 4: Your turn: a bad day
Situation: heavy rain, maintenance yes, train delayed, you miss the appointment.

**Question 8:** Compute it by hand:
$$P(\text{heavy}) \times P(\text{yes} \mid \text{heavy}) \times P(\text{delayed} \mid \text{heavy}, \text{yes}) \times P(\text{miss} \mid \text{delayed})$$
$$= 0.1 \times 0.1 \times 0.6 \times 0.4 = \mathbf{0.0024}$$

In [4]:
p_bad = (rain.get_value(rain="heavy") *
         maintenance.get_value(maintenance="yes", rain="heavy") *
         train.get_value(train="delayed", rain="heavy", maintenance="yes") *
         appointment.get_value(appointment="miss", train="delayed"))
print(round(p_bad, 4))

0.0024


---
### Step 5: Observe something: the train is delayed
Before observing anything, $P(\text{rain} = \text{none}) = 0.7$. Now we observe that the train is delayed.

**Question 9:** What happens to P(rain = none)?  
**Answer:** **It goes down** (drops from 0.7 to 0.4601).

In [5]:
from pgmpy.inference import VariableElimination

inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]} (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"  {value}: {prob:.4f}")

show({"train": "delayed"})

rain
  none: 0.4601
  light: 0.3005
  heavy: 0.2394
maintenance
  yes: 0.3662
  no: 0.6338
train: delayed (observed)
appointment
  attend: 0.6000
  miss: 0.4000


---
### Step 6: Do the same inference by hand (enumeration)
Formula:
$$P(\text{Rain} \mid \text{delayed}) = \alpha \sum_{m \in \{\text{yes}, \text{no}\}} P(\text{Rain}, m, \text{delayed})$$

**Question 10 (unnormalized values):**
- **none:** $0.7 \times (0.4 \times 0.2 + 0.6 \times 0.1) = 0.7 \times 0.14 = \mathbf{0.098}$
- **light:** $0.2 \times (0.2 \times 0.4 + 0.8 \times 0.3) = 0.2 \times 0.32 = \mathbf{0.064}$
- **heavy:** $0.1 \times (0.1 \times 0.6 + 0.9 \times 0.5) = 0.1 \times 0.51 = \mathbf{0.051}$

**Question 11 (sum):**
$$0.098 + 0.064 + 0.051 = \mathbf{0.213}$$

**Question 12 (normalized none):**
$$P(\text{rain} = \text{none} \mid \text{delayed}) = \frac{0.098}{0.213} = \mathbf{0.4601}$$

**Question 13:** Your hand answer and the code are the same: 0.4601. Why?  
**Answer:** The code does exact inference: the same sums and normalization, in a faster order.

In [6]:
# Check hand enumeration values with Python
unnorm_none = 0.7 * (0.4 * 0.2 + 0.6 * 0.1)
unnorm_light = 0.2 * (0.2 * 0.4 + 0.8 * 0.3)
unnorm_heavy = 0.1 * (0.1 * 0.6 + 0.9 * 0.5)
total = unnorm_none + unnorm_light + unnorm_heavy

print(f"none:  {unnorm_none:.3f} -> normalized: {unnorm_none / total:.4f}")
print(f"light: {unnorm_light:.3f} -> normalized: {unnorm_light / total:.4f}")
print(f"heavy: {unnorm_heavy:.3f} -> normalized: {unnorm_heavy / total:.4f}")
print(f"total sum: {total:.3f}")

none:  0.098 -> normalized: 0.4601
light: 0.064 -> normalized: 0.3005
heavy: 0.051 -> normalized: 0.2394
total sum: 0.213


---
### Step 7: Look at the Appointment result
In Step 5 output, $P(\text{appointment} = \text{miss} \mid \text{train} = \text{delayed})$ is exactly 0.4000.

**Question 14:** Why exactly 0.4?  
**Answer:** Appointment depends only on Train, and Train is known, so we read it straight from the table.

---
### Step 8: Reasoning upward: you missed the appointment
Observation: you missed the appointment.

In [7]:
show({"appointment": "miss"})

rain
  none: 0.6065
  light: 0.2392
  heavy: 0.1544
maintenance
  yes: 0.3441
  no: 0.6559
train
  on time: 0.4802
  delayed: 0.5198
appointment: miss (observed)


**Question 15:** What is P(rain = heavy | appointment = miss)?  
**Answer:** **0.1544**

**Question 16:** Heavy rain went from 0.1 to 0.15. Why?  
**Answer:** Missing makes a delay more likely, and a delay makes rain more likely.

---
### Step 9: Two observations at once
Observations: the train is delayed AND the rain is heavy.

In [8]:
show({"train": "delayed", "rain": "heavy"})

rain: heavy (observed)
maintenance
  yes: 0.1176
  no: 0.8824
train: delayed (observed)
appointment
  attend: 0.6000
  miss: 0.4000


**Question 17:** Check it by hand:
$$P(\text{maintenance} = \text{yes} \mid \text{delayed}, \text{heavy}) = \frac{0.1 \times 0.6}{0.1 \times 0.6 + 0.9 \times 0.5} = \frac{0.06}{0.06 + 0.45} = \frac{0.06}{0.51} = \mathbf{0.1176}$$

**Question 18:** With heavy rain alone, P(maintenance = yes) = 0.1. Adding "delayed" only moved it to 0.1176. Why such a small change?  
**Answer:** With heavy rain, maintenance barely changes the delay chance (0.6 vs 0.5), so the delay is weak evidence for maintenance.

---
### Step 10: Summary and Blackboard Answers

**Exit Question:** Which statement is true?  
**Answer:** Evidence at one node can change beliefs at nodes above and below it.

---
### Short Answers in My Own Words (Questions 13, 14, 16, 18)

- **Question 13 (Why hand answer and code are the same: 0.4601):**  
  Both the hand calculation and the code do the exact same math. The code doesn't guess or use random samples. It uses Variable Elimination, which calculates the exact same sums and divides by the exact same total, just in a faster order.

- **Question 14 (Why is P(appointment = miss | train = delayed) exactly 0.4):**  
  In our Bayesian network, Train is the only parent of Appointment. Since we already know the train is delayed, the appointment doesn't depend on rain or maintenance anymore. We just read 0.4 directly from the table for a delayed train.

- **Question 16 (Why heavy rain went from 0.1 to 0.1544):**  
  Missing the appointment makes it much more likely that the train was delayed (it jumps from ~21% to ~52%). And since train delays are usually caused by bad weather like heavy rain, that higher delay probability flows backwards to make heavy rain more likely too (it goes from 10% to 15.4%).

- **Question 18 (Why such a small change: 0.1 to 0.1176):**  
  With heavy rain, maintenance barely changes the chance of a delay (0.6 vs 0.5), so the delay is weak evidence for maintenance.